In [69]:
import math

class Value:
    def __init__(self,data,_children=(),_op='',label=''):
        self.data= data
        self.grad=0.0
        self._backward= lambda:None
        self.label= label
        self._prev= set(_children)
        self._op=_op
    
    def __repr__(self):
        return f"Value(data={self.data})"
    
    def __add__(self,other):
        other= other if isinstance(other,Value) else Value(other)
        out= Value(self.data + other.data, (self,other),'+')
        
        def _backward():
            self.grad+= out.grad
            other.grad+= out.grad
        
        out._backward= _backward
        return out
    
    def __mul__(self, other):
        other= other if isinstance(other,Value) else Value(other)
        out= Value(self.data * other.data, (self,other),'*')
        
        def _backward():
            self.grad+= other.data*out.grad
            other.grad+= self.data* out.grad
        
        out._backward= _backward
        return out
    
    def __pow__(self, other):
        assert isinstance(other,(int,float))
        out = Value(self.data**other, (self,),f'**{other}')
        
        def _backward():
            self.grad= (other* (self.data**(other-1))) * out.grad
        
        out._backward= _backward
        return out
    
    def __rmul__(self,other):
        return self*other
    
    def __truediv__(self, other):
        return self* other**-1
    
    def __neg__(self):
        return self*-1
    
    def __sub__(self, other):
        return self + (-other)
    
    def tanh(self):
        x= self.data
        t= (math.exp(2*x)-1)/(math.exp(2*x)+1)
        out= Value(t, (self,),'tanh')
        
        def _backward():
            self.grad+= (1- t**2)* out.grad
        
        out._backward= _backward
        return out
    
    def exp(self):
        x= self.data
        e= (math.exp(x))
        out= Value(e, (self,),'exp')
        
        def _backward():
            self.grad+= e * out.grad
        
        out._backward= _backward
        return out
    

In [70]:
a= Value(2.0,label='a')
b= Value(-3.0,label='b')
c= Value(10.0,label='c')

d= a*b + c
d.label= 'd'
e= a*b
e.label= 'e'
f= Value(-2.0,label='f')
L= d*f
L.label= 'L'

In [78]:
a/b

Value(data=-0.6666666666666666)

In [79]:
import random

class Neuron:
    def __init__(self,nin):
        self.w= [Value(random.uniform(-1,1)) for _ in range()]
        self.b= Value(random.uniform(-1,1))
    
    def __call__(self, x):
        act= sum((wi*xi for wi,xi in zip(self.w,x)),self.b)
        out= act.tanh()
        return out
        